# 04 — Fundamental Analysis Engine

Phase 4 of the Indian Equity AI project plan.

For each of the 5 stocks:
1. Download income statement, balance sheet, cash flow (Yahoo Finance)
2. Compute ratios: ROE, ROCE, Debt/Equity, Current Ratio, Interest Coverage
3. Compute growth metrics (revenue/EBITDA/EPS/FCF YoY and CAGR)
4. Pull snapshot valuation ratios (P/E, P/B, P/S, EV/EBITDA, dividend yield) and ownership %
5. Combine into a single Fundamental Score (0-100)

**Data-source note:** bank income statements (HDFCBANK) don't report EBIT/EBITDA or a
current-assets/current-liabilities split, so ROCE/current ratio/interest coverage are
NaN for it -- this is expected, not a bug. The Fundamental Score itself only depends on
ROE, Debt/Equity, revenue growth, and P/E, all of which are available for every stock,
including HDFCBANK.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.fundamentals.statements import download_statements
from src.fundamentals.ratios import (
    add_growth_metrics,
    build_annual_fundamentals,
    cagr,
    snapshot_valuation_and_ownership,
)
from src.fundamentals.score import fundamental_score

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
UNIVERSE = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "ITC"]

## Step 1-3: Download statements, compute ratios and growth metrics, per stock

In [2]:
annual_by_symbol = {}
snapshot_by_symbol = {}

for symbol in UNIVERSE:
    statements = download_statements(symbol)
    annual = build_annual_fundamentals(statements)
    annual = add_growth_metrics(annual)
    annual_by_symbol[symbol] = annual
    snapshot_by_symbol[symbol] = snapshot_valuation_and_ownership(statements["info"])
    print(symbol, "fiscal years:", [d.date() for d in annual.index])

RELIANCE fiscal years: [datetime.date(2022, 3, 31), datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


TCS fiscal years: [datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


INFY fiscal years: [datetime.date(2022, 3, 31), datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


D:\Practise\Stock Market Analysis\src\fundamentals\ratios.py:69: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  annual[f"{col}_growth_yoy"] = annual[col].pct_change()


HDFCBANK fiscal years: [datetime.date(2022, 3, 31), datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


ITC fiscal years: [datetime.date(2022, 3, 31), datetime.date(2023, 3, 31), datetime.date(2024, 3, 31), datetime.date(2025, 3, 31), datetime.date(2026, 3, 31)]


In [3]:
annual_by_symbol["RELIANCE"][
    ["revenue", "net_income", "roe", "roce", "debt_to_equity", "current_ratio", "revenue_growth_yoy"]
]

,revenue,net_income,roe,roce,debt_to_equity,current_ratio,revenue_growth_yoy
2022-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-03-31,8.778350e+12,6.670200e+11,0.093176,0.109730,0.467111,1.074677,NaN
2024-03-31,9.010640e+12,6.962100e+11,0.087741,0.112791,0.436232,1.183037,0.026462
2025-03-31,9.646930e+12,6.964800e+11,0.082600,0.107613,0.438301,1.100351,0.070615
2026-03-31,1.057219e+13,8.077500e+11,0.089350,0.115153,0.440251,1.097912,0.095912


In [4]:
annual_by_symbol["HDFCBANK"][
    ["revenue", "net_income", "roe", "roce", "debt_to_equity", "current_ratio", "revenue_growth_yoy"]
]

,revenue,net_income,roe,roce,debt_to_equity,current_ratio,revenue_growth_yoy
2022-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-03-31,1.205375e+12,4.954469e+11,0.170128,NaN,1.082272,NaN,NaN
2024-03-31,1.604968e+12,6.226566e+11,0.089625,NaN,1.164988,NaN,0.331510
2025-03-31,1.838641e+12,6.735083e+11,0.087732,NaN,0.954306,NaN,0.145593
2026-03-31,1.925668e+12,7.047934e+11,0.086294,NaN,0.813434,NaN,0.047332


## Step 3b: CAGR over the available fiscal-year window

In [5]:
cagr_rows = []
for symbol, annual in annual_by_symbol.items():
    cagr_rows.append(
        {
            "symbol": symbol,
            "revenue_cagr": cagr(annual["revenue"]),
            "eps_cagr": cagr(annual["eps_diluted"]),
            "fcf_cagr": cagr(annual["free_cash_flow"]),
        }
    )
cagr_table = pd.DataFrame(cagr_rows).set_index("symbol")
cagr_table

,revenue_cagr,eps_cagr,fcf_cagr
symbol,,,
RELIANCE,0.063941,0.065859,NaN
TCS,0.058018,0.056944,0.072516
INFY,0.034419,0.040584,0.137847
HDFCBANK,0.169014,0.010489,0.349581
ITC,0.036465,0.022145,0.003025


## Step 4-5: Build the latest-fiscal-year snapshot table and Fundamental Score

In [6]:
rows = []
for symbol in UNIVERSE:
    latest = annual_by_symbol[symbol].iloc[-1].to_dict()
    latest.update(snapshot_by_symbol[symbol])
    latest["symbol"] = symbol
    rows.append(latest)

fundamentals_snapshot = pd.DataFrame(rows).set_index("symbol")
fundamentals_snapshot["fundamental_score"] = fundamental_score(fundamentals_snapshot)
fundamentals_snapshot[
    ["roe", "debt_to_equity", "revenue_growth_yoy", "pe_ratio", "fundamental_score"]
]

,roe,debt_to_equity,revenue_growth_yoy,pe_ratio,fundamental_score
symbol,,,,,
RELIANCE,0.089350,0.440251,0.095912,22.222223,48.522027
TCS,0.458877,0.105213,0.045812,15.128615,74.471641
INFY,0.338545,0.098815,0.045702,12.902477,72.097412
HDFCBANK,0.086294,0.813434,0.047332,16.078688,45.481739
ITC,0.285343,0.033091,0.047687,16.982323,67.126635


## Step 6: Save fundamentals

In [7]:
assert fundamentals_snapshot["fundamental_score"].notna().all(), (
    "Every stock in the universe should have a computable Fundamental Score"
)

fundamentals_snapshot.to_csv(DATA_PROCESSED / "fundamentals_snapshot.csv")

annual_long = pd.concat(
    [df.assign(symbol=symbol) for symbol, df in annual_by_symbol.items()]
).reset_index(names="fiscal_year_end")
annual_long.to_parquet(DATA_PROCESSED / "fundamentals_annual.parquet", index=False)

print("Saved:", DATA_PROCESSED / "fundamentals_snapshot.csv")
print("Saved:", DATA_PROCESSED / "fundamentals_annual.parquet")

Saved: D:\Practise\Stock Market Analysis\data\processed\fundamentals_snapshot.csv
Saved: D:\Practise\Stock Market Analysis\data\processed\fundamentals_annual.parquet
